In [2]:
import json
import re
import os
import pandas as pd
import openpyxl
import io
import tokenize
from typing import List, Dict, Optional

# --- 設定項目 ------------------
examid = 2 # 試験の番号

grading_excel_path = '評定.xlsx' # コース評定のExcelエクスポートファイルを名寄せに利用する

# Moodle の 課題＞提出＞操作＞全ての提出をダウンロードする で学習者の提出ファイルをダウンロードし、
# target_directory 内に、Moodle課題のすべての提出ファイルを置く。
# 読み込むファイルは、.py または .ipynb のみ
target_directory = './submissions' #

output_excel_name = 'extracted_code_with_userid.xlsx'
# ------------------------------

def _robustly_clean_code(full_code: str) -> str:
    if not full_code:
        return ""
    full_code = re.sub(r'^[ \t]*(?:!|%).*\n?', '', full_code, flags=re.MULTILINE)
    code_without_multiline_comments = re.sub(r'[uU]?[rR]?("""|\'\'\').*?\1', '', full_code, flags=re.DOTALL)
    code_stream = io.StringIO(code_without_multiline_comments)
    output_tokens = []
    try:
        for token in tokenize.generate_tokens(code_stream.readline):
            if token.type not in (tokenize.COMMENT, tokenize.NL, tokenize.NEWLINE, tokenize.ENCODING, tokenize.ENDMARKER):
                output_tokens.append(token)
    except Exception:
        return code_without_multiline_comments
    rebuilt_code = []
    last_row, last_col = 1, 0
    for token in output_tokens:
        start_row, start_col = token.start
        if start_row > last_row:
            rebuilt_code.append('\n' * (start_row - last_row) + ' ' * start_col)
        elif start_col > last_col:
            rebuilt_code.append(' ' * (start_col - last_col))
        rebuilt_code.append(token.string)
        last_row, last_col = token.end
    return "\n".join([line.rstrip() for line in "".join(rebuilt_code).split('\n') if line.strip()])

def extract_code_from_ipynb(filepath: str) -> Optional[str]:
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            notebook = json.load(f)
    except Exception: return None
    all_code_parts = [_robustly_clean_code("".join(c.get('source', []))) 
                      for c in notebook.get('cells', []) if c.get('cell_type') == 'code']
    return "\n".join(filter(None, all_code_parts))

def extract_code_from_py(filepath: str) -> Optional[str]:
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            return _robustly_clean_code(f.read())
    except Exception: return None

# 評定Excelから名前とIDナンバーの対応マップを作成
def get_name_to_id_map(excel_path: str) -> Dict[str, str]:
    df_grading = pd.read_excel(excel_path)
    # 「姓 名」の形式でマップを作成
    # 姓と名の間のスペースは、ファイル名に合わせて半角スペースを入れています
    mapping = {}
    for _, row in df_grading.iterrows():
        full_name = f"{row['姓']} {row['名']}"
        mapping[full_name] = str(row['IDナンバ'])
    return mapping

def process_notebooks_and_save_excel(directory: str, output_filename: str):
    data_list = []
    
    # 1. 名簿マップの取得
    name_map = get_name_to_id_map(grading_excel_path)
    
    if not os.path.exists(directory):
        print(f"エラー: ディレクトリ '{directory}' が見つかりません。")
        return

    for filename in os.listdir(directory):
        filepath = os.path.join(directory, filename)
        extracted_code = None

        if filename.endswith('.ipynb'):
            extracted_code = extract_code_from_ipynb(filepath)
        elif filename.endswith('.py'):
            extracted_code = extract_code_from_py(filepath)
        
        if extracted_code is not None:
            # 2. ファイル名から「フルネーム」を抽出 (最初のアンダースコアの前)
            # 例: "山中 雄大_12345_..." -> "山中 雄大"
            full_name_in_file = filename.split('_')[0]
            
            # 3. マップからIDナンバーを取得。なければ元の名前を保持
            user_id = name_map.get(full_name_in_file, f"Unknown({full_name_in_file})")

            data_list.append({
                'filename': filename,
                'username': user_id, # ここにIDナンバーが入る
                'extractedcode': extracted_code,
                'examid': examid
            })

    if not data_list:
        print("処理対象ファイルが見つかりませんでした。")
        return

    df = pd.DataFrame(data_list)
    df_sorted = df.sort_values(by='username')

    try:
        with pd.ExcelWriter(output_filename, engine='openpyxl') as writer:
            df_sorted.to_excel(writer, sheet_name='Sheet1', index=False)
            ws = writer.sheets['Sheet1']
            ws.column_dimensions['A'].width = 30
            ws.column_dimensions['B'].width = 15
            ws.column_dimensions['C'].width = 60
            ws.column_dimensions['D'].width = 10
            
            from openpyxl.styles import Alignment
            wrap = Alignment(wrap_text=True, vertical='top')
            center = Alignment(horizontal='center', vertical='top')
            
            for row in range(1, len(df_sorted) + 2):
                ws.row_dimensions[row].height = 113.4 if row > 1 else 15
                for col in [1, 2, 3]: ws.cell(row, col).alignment = wrap
                ws.cell(row, 4).alignment = center
            
            print(f"✅ 処理完了: {output_filename} (対象: {len(data_list)}件)")
    except Exception as e:
        print(f"エラー: {e}")

if __name__ == "__main__":
    process_notebooks_and_save_excel(target_directory, output_excel_name)

処理対象ファイルが見つかりませんでした。
